In [ ]:
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
import os

d = {'NAME': [], 'PRICE': [], 'LINK': []}

# UPDATED: Using your actual folder name
folder_name = "amazon web scrapping data"

for file in os.listdir(folder_name):
    try:
        with open(f"{folder_name}/{file}", encoding='utf-8') as f:
            html_doc = f.read()

        soup = BeautifulSoup(html_doc, 'html.parser')
        
        products = soup.find_all('div', attrs={'data-component-type': 's-search-result'})
        
        for item in products:
            name_elem = item.find("span", attrs={'class': 'a-size-medium a-color-base a-text-normal'})
            name = name_elem.text if name_elem else np.nan
            
            price_elem = item.find("span", attrs={'class': 'a-price-whole'})
            price = price_elem.text if price_elem else np.nan
            
            link_elem = item.find("a", attrs={'class': 'a-link-normal s-line-clamp-2 s-link-style a-text-normal'})
            link = "https://www.amazon.in." + link_elem['href'] if link_elem and link_elem.has_attr('href') else np.nan
            
            d['NAME'].append(name)
            d['PRICE'].append(price)
            d['LINK'].append(link)
            
    except Exception as e:
        print(f"Error reading {file}: {e}") # Added printing so errors aren't hidden

df = pd.DataFrame(data=d)

df['NAME'] = df['NAME'].replace('N/A', np.nan)
df['PRICE'] = df['PRICE'].replace('N/A', np.nan)
df.dropna(subset=['NAME', 'PRICE'], inplace=True)

df['PRICE'] = df['PRICE'].astype(str).str.replace(',', '', regex=False).str.replace('₹', '', regex=False)
df['PRICE ₹'] = pd.to_numeric(df['PRICE'], errors='coerce')
df.drop('PRICE', axis=1, inplace=True)
df.dropna(subset=['PRICE ₹'], inplace=True)

df['MODEL NAME'] = df['NAME'].str.split(r'(?=[|(])', n=1).str[0].str.strip()
df['MODEL SPECS'] = df['NAME'].str.split(r'(?=[|(])', n=1).str[1].str.strip()
df['COMPANY'] = df['MODEL NAME'].str.split().str[0].str.upper()

df = df[df['NAME'].str.contains(r'\bGB\b', na=False, case=False)]
df_final = df[df['PRICE ₹'] > 4000].copy()

df_final['COMPANY'] = df_final['COMPANY'].replace({
    'LATEST' : 'F24 PRO',
    'KIDS' : 'UNKNOWN'
})

df_final = df_final[['COMPANY', 'MODEL NAME', 'MODEL SPECS', 'PRICE ₹', 'LINK']]
df_final.head()


AttributeError: Can only use .str accessor with string values, not floating